In [9]:
import os
import re
import json

import requests
import psycopg2
import pandas as pd

# Configurações da API Local (mesma x-api-key usada em processos-por-cpf.ipynb / get-partners.ipynb)
API_URL = "http://10.210.10.19:3003/reputational"
API_KEY = "UgdRwHpekKRWafd+gA0Q1I72iCeQuKAqArrMEkU43f6UdgOJDsUnqoMKDB+2hhk8st9DNW9gSozSUdvKGI2w89RMMlMGEAEb1hYILnjroouAAMvAvZygGORoaX3DYQi4Dj8KX0mtHRcYRS7XW76BOxpijFVUa3IdhYLhZJIL1uo="
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json",
}
WEBHOOK_URL = "https://webhook.site/0bf0f18b-533a-411b-ac4f-9f46478a9e7b"

# Configurações do Postgres (kronoos-bases)
DB_PARAMS = {
    "host":     os.environ.get("PGHOST",     "10.210.10.15"),
    "port":     os.environ.get("PGPORT",     "5432"),
    "user":     os.environ.get("PGUSER",     "postgres"),
    "password": os.environ.get("PGPASSWORD", "sofkronoos@@2020"),
    "dbname":   os.environ.get("PGDATABASE", "kronoos-bases"),
}

SCRIPT_NAME = "reputacional-listas-ppe"

# (nome da aba no xlsx original, slug usado nos arquivos de input/output)
PAGINAS = [
    ("Ouro Bruto PJ", "ouro_bruto_pj"),
    ("Ouro Bruto PF", "ouro_bruto_pf"),
    ("Ouro Fino PF", "ouro_fino_pf"),
    ("Ouro Fino PJ", "ouro_fino_pj"),
]

conn = psycopg2.connect(**DB_PARAMS)
print("Conectado ao Postgres — banco 'kronoos-bases'.")
print("Ambiente configurado com sucesso.")


Conectado ao Postgres — banco 'kronoos-bases'.
Ambiente configurado com sucesso.


In [10]:
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

RESPONSES_DIR = os.path.join("..", "responses", SCRIPT_NAME)
CACHE_DIR = os.path.join(RESPONSES_DIR, "cache")
for sub in ("socioambientais", "internacionais", "ppe"):
    os.makedirs(os.path.join(CACHE_DIR, sub), exist_ok=True)


def normalize_doc(document):
    """Remove tudo que não for dígito — aceita 000.000.000-00/00.000.000/0001-00 ou já limpo."""
    return re.sub(r"\D", "", str(document))


def cache_get(fonte, doc_key):
    cache_file = os.path.join(CACHE_DIR, fonte, f"{doc_key}.json")
    if os.path.exists(cache_file):
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def cache_set(fonte, doc_key, payload):
    cache_file = os.path.join(CACHE_DIR, fonte, f"{doc_key}.json")
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


THIN_BORDER = Border(
    left=Side(style="thin", color="BFBFBF"),
    right=Side(style="thin", color="BFBFBF"),
    top=Side(style="thin", color="BFBFBF"),
    bottom=Side(style="thin", color="BFBFBF"),
)
HEADER_BORDER = Border(
    left=Side(style="thin", color="BFBFBF"),
    right=Side(style="thin", color="BFBFBF"),
    top=Side(style="thin", color="BFBFBF"),
    bottom=Side(style="medium", color="808080"),
)


def format_sheet(ws, df, col_colors):
    """Cabeçalho colorido, sem preenchimento nas linhas de dados, bordas, largura
    automática de coluna, congelamento do cabeçalho e autofiltro."""
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    data_align = Alignment(horizontal="left", vertical="center", wrap_text=False)

    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 22

    for col_idx, col_name in enumerate(df.columns, start=1):
        header_hex, _ = col_colors.get(col_name, ("D9D9D9", "F5F5F5"))

        header_cell = ws.cell(row=1, column=col_idx)
        header_cell.fill = PatternFill("solid", fgColor=header_hex)
        header_cell.font = Font(bold=True, color="3B3B3B", size=10)
        header_cell.alignment = header_align
        header_cell.border = HEADER_BORDER

        valores = df[col_name] if len(df) else []
        max_len = max([len(str(col_name))] + [len(str(v)) for v in valores]) if len(df) else len(str(col_name))
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 4, 12), 60)

        for row_idx in range(2, len(df) + 2):
            data_cell = ws.cell(row=row_idx, column=col_idx)
            data_cell.alignment = data_align
            data_cell.border = THIN_BORDER

    ws.auto_filter.ref = ws.dimensions


print(f"Funções e esquema de cores definidos. Cache em: {CACHE_DIR}")


Funções e esquema de cores definidos. Cache em: ../responses/reputacional-listas-ppe/cache


In [11]:
def fetch_socioambientais(documento):
    """Consulta a API /reputational e devolve a lista data[0]['socioambientais'].
    Retorna None em caso de erro na chamada (não achar o doc não é erro — devolve [])."""
    doc_key = normalize_doc(documento)
    cached = cache_get("socioambientais", doc_key)
    if cached is not None:
        print("  -> Cache encontrado (socioambientais), pulando chamada à API")
        return cached.get("data", [{}])[0].get("socioambientais", []) if cached.get("data") else []

    body = {
        "identifier": 1,
        "webhook_url": WEBHOOK_URL,
        "document": doc_key,
        "async": False,
    }
    try:
        response = requests.post(API_URL, headers=HEADERS, json=body)
        response.raise_for_status()
        payload = response.json()
        cache_set("socioambientais", doc_key, payload)
    except requests.exceptions.RequestException as e:
        print(f"  -> Erro na chamada da API /reputational: {e}")
        return None

    data = payload.get("data", [])
    return data[0].get("socioambientais", []) if data else []


print("Função de consulta Socioambientais (API) carregada.")


Função de consulta Socioambientais (API) carregada.


In [12]:
SQL_INTERNACIONAIS = """
    SELECT nome, regiao, titulo, fonte_noticia, data_noticia
    FROM lista_internacionais_atual
    WHERE nome ILIKE %s AND (titulo ILIKE %s OR titulo ILIKE %s)
"""


def fetch_internacionais(nome, documento):
    """Busca em lista_internacionais_atual por nome (ILIKE), restrito a titulo
    contendo 'csnu' ou 'ofac'. Retorna None em caso de erro na query."""
    doc_key = normalize_doc(documento)
    cached = cache_get("internacionais", doc_key)
    if cached is not None:
        print("  -> Cache encontrado (internacionais), pulando consulta ao banco")
        return cached

    try:
        with conn.cursor() as cur:
            cur.execute(SQL_INTERNACIONAIS, (f"%{nome}%", "%csnu%", "%ofac%"))
            cols = [c.name for c in cur.description]
            rows = [dict(zip(cols, row)) for row in cur.fetchall()]
        cache_set("internacionais", doc_key, rows)
        return rows
    except Exception as e:
        conn.rollback()
        print(f"  -> Erro na consulta a lista_internacionais_atual: {e}")
        return None


print("Função de consulta Listas Internacionais (Postgres) carregada.")


Função de consulta Listas Internacionais (Postgres) carregada.


In [13]:
PPE_COLS = ["nome_cpf", "cpf", "sexo", "envolvimento", "atividade", "regiao", "estado", "pep_relacionado"]

# DISTINCT porque lista_ppe_atual tem linhas duplicadas de verdade (mesmo conteúdo,
# ids diferentes) — sobretudo para relacionados, que às vezes aparecem dezenas de
# vezes com os mesmos dados. Sem DISTINCT o relatório mostrava a mesma resposta
# repetida várias vezes.
SQL_PPE_POR_CPF = f"""
    SELECT DISTINCT {', '.join(f'p.{c}' for c in PPE_COLS)}, t.nome_cpf AS nome_titular
    FROM lista_ppe_atual p
    LEFT JOIN lista_ppe_atual t
        ON t.cpf = p.pep_relacionado AND t.envolvimento ILIKE %s
    WHERE p.cpf = %s
"""
SQL_PPE_RELACIONADOS = f"""
    SELECT DISTINCT {', '.join(PPE_COLS)}
    FROM lista_ppe_atual
    WHERE pep_relacionado = %s
"""


def _run_ppe_query(sql, params):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [c.name for c in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]


def fetch_ppe(documento):
    """Busca em lista_ppe_atual pelo documento (cpf). Se o documento for um PPE
    Titular, busca também todos os relacionados (pep_relacionado = documento). Se
    for um PPE Relacionado, traz junto o nome do titular a quem ele é relacionado
    (nome_titular, resolvido via pep_relacionado). Retorna None em caso de erro."""
    doc_key = normalize_doc(documento)
    cached = cache_get("ppe", doc_key)
    if cached is not None:
        print("  -> Cache encontrado (ppe), pulando consulta ao banco")
        return cached

    try:
        rows = _run_ppe_query(SQL_PPE_POR_CPF, ("%titular%", doc_key))
        if any("titular" in (r.get("envolvimento") or "").lower() for r in rows):
            rows += _run_ppe_query(SQL_PPE_RELACIONADOS, (doc_key,))
        cache_set("ppe", doc_key, rows)
        return rows
    except Exception as e:
        conn.rollback()
        print(f"  -> Erro na consulta a lista_ppe_atual: {e}")
        return None


print("Função de consulta PPE (Postgres) carregada.")


Função de consulta PPE (Postgres) carregada.


In [14]:
def process_pagina(sheet_name, slug):
    """Lê input/<slug>_documentos.json e roda as buscas para cada documento.
    Cada documento gera sempre uma linha por tópico, mesmo sem resultado (campos da
    fonte em branco) — exceto PPE, que só é consultada para documentos PF (a tabela
    lista_ppe_atual não se aplica a CNPJ).
    Devolve (socio_rows, intl_rows, ppe_rows, erros, total_documentos)."""
    input_path = os.path.join("..", "input", f"{slug}_documentos.json")
    with open(input_path, "r", encoding="utf-8") as f:
        document_list = json.load(f)

    tipo = sheet_name.split()[-1]  # "Ouro Bruto PJ" -> "PJ"
    print(f"\n=== {sheet_name} — {len(document_list)} documento(s) ===")

    socio_rows, intl_rows, ppe_rows, erros = [], [], [], []

    for entry in document_list:
        nome = entry.get("nome", "")
        documento = entry.get("documento", "")
        print(f"Processando: {nome} ({documento})")
        base = {"Nome": nome, "Documento": documento, "Tipo": tipo}

        socio = fetch_socioambientais(documento)
        if socio is None:
            erros.append((nome, documento, "Socioambientais (API)", "Erro ao consultar /reputational"))
        elif not socio:
            socio_rows.append({
                **base,
                "Título": None, "Citação": None, "Fonte": None, "Data": None, "URL": None,
                "Atividade": None, "Envolvimento": None, "Suspeita": None, "Região": None, "Estado": None,
            })
        else:
            for item in socio:
                socio_rows.append({
                    **base,
                    "Título": item.get("titulo"),
                    "Citação": item.get("citacao"),
                    "Fonte": item.get("fonte"),
                    "Data": item.get("data"),
                    "URL": item.get("url"),
                    "Atividade": item.get("atividade"),
                    "Envolvimento": item.get("envolvimento"),
                    "Suspeita": item.get("suspeita"),
                    "Região": item.get("regiao"),
                    "Estado": item.get("estado"),
                })

        intl = fetch_internacionais(nome, documento)
        if intl is None:
            erros.append((nome, documento, "Listas Internacionais (Banco)", "Erro ao consultar lista_internacionais_atual"))
        elif not intl:
            intl_rows.append({
                **base,
                "Nome na Lista": None, "Região": None, "Lista (Título)": None,
                "Fonte da Notícia": None, "Data da Notícia": None,
            })
        else:
            for item in intl:
                intl_rows.append({
                    **base,
                    "Nome na Lista": item.get("nome"),
                    "Região": item.get("regiao"),
                    "Lista (Título)": item.get("titulo"),
                    "Fonte da Notícia": item.get("fonte_noticia"),
                    "Data da Notícia": item.get("data_noticia"),
                })

        if tipo == "PF":
            ppe = fetch_ppe(documento)
            if ppe is None:
                erros.append((nome, documento, "PPE (Banco)", "Erro ao consultar lista_ppe_atual"))
            elif not ppe:
                ppe_rows.append({
                    **base,
                    "Nome (PPE)": None, "CPF": None, "Sexo": None, "Envolvimento PPE": None,
                    "Atividade": None, "Região": None, "Estado": None,
                    "PPE Relacionado": None, "Nome do Titular": None,
                })
            else:
                for item in ppe:
                    ppe_rows.append({
                        **base,
                        "Nome (PPE)": item.get("nome_cpf"),
                        "CPF": item.get("cpf"),
                        "Sexo": item.get("sexo"),
                        "Envolvimento PPE": item.get("envolvimento"),
                        "Atividade": item.get("atividade"),
                        "Região": item.get("regiao"),
                        "Estado": item.get("estado"),
                        "PPE Relacionado": item.get("pep_relacionado"),
                        "Nome do Titular": item.get("nome_titular"),
                    })
        # PJ: lista_ppe_atual é só PF — não consulta e não gera linha em PPE.

    return socio_rows, intl_rows, ppe_rows, erros, len(document_list)


print("Função de processamento por página carregada.")


Função de processamento por página carregada.


In [15]:
# Cores pastéis por grupo de coluna: (header_hex, data_hex)
# Amarelo → identidade do input (Nome, Documento, Tipo)
COLORS_SOCIOAMBIENTAIS = {
    "Nome":         ("FFD966", "FFFCE8"),
    "Documento":    ("FFD966", "FFFCE8"),
    "Tipo":         ("FFD966", "FFFCE8"),
    "Título":       ("9DC3E6", "EBF3FB"),
    "Citação":      ("9DC3E6", "EBF3FB"),
    "URL":          ("9DC3E6", "EBF3FB"),
    "Fonte":        ("F4B183", "FEF3EA"),
    "Data":         ("F4B183", "FEF3EA"),
    "Atividade":    ("A9D18E", "EEF5E9"),
    "Envolvimento": ("A9D18E", "EEF5E9"),
    "Suspeita":     ("A9D18E", "EEF5E9"),
    "Região":       ("C9A0DC", "F5EEF8"),
    "Estado":       ("C9A0DC", "F5EEF8"),
}
COLORS_INTERNACIONAIS = {
    "Nome":             ("FFD966", "FFFCE8"),
    "Documento":        ("FFD966", "FFFCE8"),
    "Tipo":             ("FFD966", "FFFCE8"),
    "Nome na Lista":    ("9DC3E6", "EBF3FB"),
    "Lista (Título)":   ("9DC3E6", "EBF3FB"),
    "Região":           ("A9D18E", "EEF5E9"),
    "Fonte da Notícia": ("F4B183", "FEF3EA"),
    "Data da Notícia":  ("F4B183", "FEF3EA"),
}
COLORS_PPE = {
    "Nome":             ("FFD966", "FFFCE8"),
    "Documento":        ("FFD966", "FFFCE8"),
    "Tipo":             ("FFD966", "FFFCE8"),
    "Nome (PPE)":       ("9DC3E6", "EBF3FB"),
    "CPF":              ("9DC3E6", "EBF3FB"),
    "Sexo":             ("9DC3E6", "EBF3FB"),
    "Envolvimento PPE": ("A9D18E", "EEF5E9"),
    "Atividade":        ("A9D18E", "EEF5E9"),
    "Região":           ("C9A0DC", "F5EEF8"),
    "Estado":           ("C9A0DC", "F5EEF8"),
    "PPE Relacionado":  ("F4B183", "FEF3EA"),
    "Nome do Titular":  ("F4B183", "FEF3EA"),
}

COLS_SOCIOAMBIENTAIS = list(COLORS_SOCIOAMBIENTAIS.keys())
COLS_INTERNACIONAIS = list(COLORS_INTERNACIONAIS.keys())
COLS_PPE = list(COLORS_PPE.keys())


def write_relatorio(slug, socio_rows, intl_rows, ppe_rows, erros, total_documentos):
    os.makedirs(RESPONSES_DIR, exist_ok=True)

    df_socio = pd.DataFrame(socio_rows, columns=COLS_SOCIOAMBIENTAIS) if socio_rows else pd.DataFrame(columns=COLS_SOCIOAMBIENTAIS)
    df_intl = pd.DataFrame(intl_rows, columns=COLS_INTERNACIONAIS) if intl_rows else pd.DataFrame(columns=COLS_INTERNACIONAIS)
    df_ppe = pd.DataFrame(ppe_rows, columns=COLS_PPE) if ppe_rows else pd.DataFrame(columns=COLS_PPE)

    output_file = os.path.join(RESPONSES_DIR, f"relatorio_{slug}.xlsx")
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_socio.to_excel(writer, sheet_name="Socioambientais", index=False)
        df_intl.to_excel(writer, sheet_name="Listas Internacionais", index=False)
        df_ppe.to_excel(writer, sheet_name="PPE", index=False)
        format_sheet(writer.sheets["Socioambientais"], df_socio, COLORS_SOCIOAMBIENTAIS)
        format_sheet(writer.sheets["Listas Internacionais"], df_intl, COLORS_INTERNACIONAIS)
        format_sheet(writer.sheets["PPE"], df_ppe, COLORS_PPE)

    erros_path = os.path.join(RESPONSES_DIR, f"erros_{slug}.txt")
    with open(erros_path, "w", encoding="utf-8") as f:
        f.write(f"Erros de consulta — {len(erros)} ocorrência(s)\n")
        f.write("=" * 50 + "\n\n")
        for nome_err, doc_err, etapa, motivo in erros:
            f.write(f"{nome_err} | {doc_err} | {etapa} | {motivo}\n")

    print(f"\n--- Resumo: {slug} ---")
    print(f"Documentos processados      : {total_documentos}")
    print(f"Linhas em Socioambientais    : {len(df_socio)}")
    print(f"Linhas em Listas Internacionais : {len(df_intl)}")
    print(f"Linhas em PPE                : {len(df_ppe)}")
    print(f"Erros                        : {len(erros)}")
    print(f"Arquivo salvo em: {output_file}")
    if erros:
        print(f"Erros salvos em: {erros_path}")


print("Esquema de cores e função de escrita do relatório carregados.")


Esquema de cores e função de escrita do relatório carregados.


In [16]:
resumo_geral = []

for sheet_name, slug in PAGINAS:
    socio_rows, intl_rows, ppe_rows, erros, total_documentos = process_pagina(sheet_name, slug)
    write_relatorio(slug, socio_rows, intl_rows, ppe_rows, erros, total_documentos)
    resumo_geral.append({
        "Página": sheet_name,
        "Documentos": total_documentos,
        "Socioambientais": len(socio_rows),
        "Listas Internacionais": len(intl_rows),
        "PPE": len(ppe_rows),
        "Erros": len(erros),
    })

conn.close()

print(f"\n{'='*60}")
print("RESUMO GERAL DA EXECUÇÃO")
print(f"{'='*60}")
display(pd.DataFrame(resumo_geral))



=== Ouro Bruto PJ — 26 documento(s) ===
Processando: A R WEBER (05.742.530/0001-29)
  -> Cache encontrado (socioambientais), pulando chamada à API
  -> Cache encontrado (internacionais), pulando consulta ao banco
Processando: COOPERATIVA DOS GARIMPEIROS DO RIO MADEIRA - COOGARIMA (05.972.820/0001-69)
  -> Cache encontrado (socioambientais), pulando chamada à API
  -> Cache encontrado (internacionais), pulando consulta ao banco
Processando: MINERACAO ABDALA LTDA (08.838.089/0001-71)
  -> Cache encontrado (socioambientais), pulando chamada à API
  -> Cache encontrado (internacionais), pulando consulta ao banco
Processando: Coogavepe Cooperativa dos Garimpeiros do Vale do Rio Peixoto (09.521.470/0001-75)
  -> Cache encontrado (socioambientais), pulando chamada à API
  -> Cache encontrado (internacionais), pulando consulta ao banco
Processando: Cooperativa de Extracao Mineral do Vale do Tapajos - Coopemvat (10.221.315/0001-12)
  -> Cache encontrado (socioambientais), pulando chamada à API

,Página,Documentos,Socioambientais,Listas Internacionais,PPE,Erros
0,Ouro Bruto PJ,26,45,26,0,0
1,Ouro Bruto PF,1022,1032,1022,1034,0
2,Ouro Fino PF,55,64,55,55,0
3,Ouro Fino PJ,33,37,33,0,0
